# MyMovie Studio · Colab ComfyUI client · 0.1.21
**Enable Internet, choose a CPU or GPU runtime, then choose Run all. No edits, Secrets, copied hashes or manual installs are needed for the default test setup.**

The notebook installs pinned ComfyUI v0.40.0 in its own session folder, reuses the runtime's PyTorch in an isolated Python environment, installs the workflow-import browser and checksum-verified cloudflared, and downloads a verified released Linux x86_64 agent + relay. **No model weights or workflow plugins are downloaded.** Install those later through Studio's workflow initialization.

The predefined public test keys match Studio → Settings → Node access. Optional environment/Secrets overrides are commented out. Use a private pair in both places for private deployments. Public agent releases require no GitHub token; optional custom release/archive settings are below. Run all cells from the top. Setup can also restore bundled helpers if the helper cell was skipped.

At the end copy the **public relay URL** and **node ID** into Studio → Connections → Colab / Kaggle; enable **Temporary Cloudflare tunnel · polling only**. ComfyUI stays on loopback. Quick Tunnels use polling, not SSE; their addresses change after a restart. Select Inventory, pin its native runtime fingerprint in Settings → App Runner, then Review initialization and Initialize your workflow. Collect results into `.mymovie` before ending the session.

Automatic mode uses up to two independent CUDA workers on Kaggle and one on Colab; without a GPU it starts one CPU worker. Set `DEVICE = "cpu"` to force CPU, or `DEVICE = "cuda"` to require the requested GPUs. GPU memory is not pooled. CPU is suitable for connection, initialization and small mask tests; model generation can be very slow. A healthy repeated Run all reuses the running session. The final Stop cell is disabled by default, so Run all leaves the node running. Runtime termination still ends the session; no keepalive is included.

Google's managed-runtime usage restrictions still apply; use an eligible runtime for externally controlled workers. See https://research.google.com/colaboratory/faq.html .


In [ ]:
"""Verified Linux agent deployment with CPU or independent notebook GPU workers.

Stdlib bootstrap; installs base runtime/tools, never weights or workflow plugins.
"""
from __future__ import annotations

import fcntl
import hashlib
import json
import os
from pathlib import Path
import platform
import re
import shutil
import signal
import socket
import subprocess
import sys
import tarfile
import tempfile
import time
import urllib.parse
import urllib.request
import uuid

MAX_ARTIFACT = 256 * 1024 * 1024
MAX_BINARY = 128 * 1024 * 1024


class SetupError(RuntimeError):
    """Public messages are deliberately independent of network/secret exceptions."""


def prepare_runtime(binary, comfy_root, revision, work_dir, *, tunnel_source=None, tunnel_sha256=None, cpu=False):
    """Provision a base install; reviewed workflow dependencies belong to Studio."""
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise SetupError("Automatic notebook setup requires a Linux x86_64 runtime.")
    if not re.fullmatch(r"[a-f0-9]{40}", revision or ""):
        raise SetupError("The notebook must contain a pinned ComfyUI revision.")
    root, work = Path(comfy_root).resolve(), Path(work_dir).resolve()
    work.mkdir(parents=True, exist_ok=True)
    log = work / "installation.log"
    log.touch(mode=0o600); log.chmod(0o600)
    env = {k: v for k, v in os.environ.items() if not k.startswith("MYMOVIE_")}
    env.update(GIT_TERMINAL_PROMPT="0", PIP_NO_INPUT="1", DEBIAN_FRONTEND="noninteractive")

    def command(args, stage, timeout=1200):
        print(stage, flush=True)
        try:
            with log.open("a") as output:
                subprocess.run([str(a) for a in args], check=True, stdout=output,
                               stderr=subprocess.STDOUT, env=env, timeout=timeout)
        except Exception:
            raise SetupError(f"{stage} failed. Inspect {log} in this runtime, then retry Run all.") from None

    if not root.exists():
        root.parent.mkdir(parents=True, exist_ok=True)
        # Publish a complete checkout; interrupted clones cannot leave a broken target.
        with tempfile.TemporaryDirectory(prefix=".comfy-", dir=root.parent) as directory:
            checkout = Path(directory) / "repo"
            command(["git", "init", checkout], "Preparing base ComfyUI")
            command(["git", "-C", checkout, "fetch", "--depth=1",
                     "https://github.com/Comfy-Org/ComfyUI.git", revision], "Downloading pinned ComfyUI")
            command(["git", "-C", checkout, "checkout", "--detach", "FETCH_HEAD"], "Activating pinned ComfyUI")
            checkout.rename(root)
    try:
        actual = subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL, timeout=10).strip()
        dirty = subprocess.check_output(["git", "-C", str(root), "status", "--porcelain", "--untracked-files=no"], text=True, stderr=subprocess.DEVNULL, timeout=10).strip()
    except Exception:
        raise SetupError(f"Cannot verify {root}. Existing files were left untouched; choose an empty COMFY_ROOT.") from None
    if actual != revision or dirty or not (root / "main.py").is_file():
        raise SetupError(f"Existing ComfyUI at {root} conflicts with the pinned clean revision; it was left untouched.")

    requirements = Path(binary).parent / "workflow-compiler-requirements.txt"
    if not requirements.is_file():
        raise SetupError("The agent archive must include workflow-compiler-requirements.txt.")
    python = work / "env/bin/python"
    marker = work / "base-runtime.json"
    expected = {"schema": 1, "device": "cpu" if cpu else "cuda", "revision": revision, "python": sys.executable,
                "pythonVersion": sys.version, "comfyRoot": str(root),
                "requirements": hashlib.sha256((root / "requirements.txt").read_bytes() + requirements.read_bytes()).hexdigest()}
    try:
        cached = python.is_file() and json.loads(marker.read_text()) == expected
    except (OSError, ValueError):
        cached = False
    if not cached:
        marker.unlink(missing_ok=True)
        # Reuse the runtime's PyTorch without changing notebook packages.
        # Colab may omit ensurepip; inherit its existing pip instead of requiring apt.
        command([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", work / "env"], "Creating isolated Python environment")
        command([python, "-m", "pip", "install", "-r", root / "requirements.txt", "-r", requirements], "Installing ComfyUI and workflow-import dependencies")
        command([python, "-m", "playwright", "install", "--with-deps", "chromium"], "Installing workflow-import browser")
    # A marker alone is never readiness; validate the chosen device and the browser's OS libraries.
    try:
        command([python, "-c", "import sys; sys.path.insert(0, sys.argv[1]); sys.argv=sys.argv[:1]; import torch, aiohttp, folder_paths; "
                 "from playwright.sync_api import sync_playwright; " +
                 ("assert torch.ones(1).device.type == 'cpu'; " if cpu else "assert torch.cuda.is_available(), 'CUDA unavailable'; ") +
                 "p=sync_playwright().start(); browser=p.chromium.launch(headless=True, args=['--no-sandbox']); browser.close(); p.stop()", root],
                f"Verifying {'CPU' if cpu else 'CUDA'} and workflow-import browser", timeout=90)
    except SetupError:
        marker.unlink(missing_ok=True)
        raise
    temporary = marker.with_suffix(".tmp")
    temporary.write_text(json.dumps(expected)); temporary.replace(marker)
    tunnel = None
    if tunnel_source:
        private_url(tunnel_source)
        if not re.fullmatch(r"[a-f0-9]{64}", tunnel_sha256 or ""):
            raise SetupError("A pinned cloudflared SHA256 is required.")
        tunnel = work / "cloudflared"
        if not tunnel.is_file() or hashlib.sha256(tunnel.read_bytes()).hexdigest() != tunnel_sha256:
            print("Downloading verified cloudflared", flush=True)
            temporary = tunnel.with_suffix(".part")
            try:
                with urllib.request.urlopen(tunnel_source, timeout=60) as response, temporary.open("wb") as output:
                    size, digest = 0, hashlib.sha256()
                    while chunk := response.read(1024 * 1024):
                        size += len(chunk)
                        if size > MAX_BINARY:
                            raise SetupError("cloudflared download exceeds the size limit.")
                        digest.update(chunk); output.write(chunk)
                if digest.hexdigest() != tunnel_sha256:
                    raise SetupError("cloudflared checksum mismatch; executable was not installed.")
                temporary.chmod(0o700); temporary.replace(tunnel)
            except SetupError:
                raise
            except Exception:
                raise SetupError("cloudflared download failed. Check Internet access and retry Run all.") from None
            finally:
                temporary.unlink(missing_ok=True)
        tunnel.chmod(0o700)
    print("Base runtime ready. Models and workflow plugins will be installed from Studio.", flush=True)
    return {"python": str(python), "cloudflared": str(tunnel) if tunnel else None}


def secret(host: str, label: str) -> str:
    try:
        if host == "colab":
            from google.colab import userdata
            value = userdata.get(label)
        elif host == "kaggle":
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(label)
        else:
            raise SetupError("Choose colab or kaggle for notebook Secrets.")
    except Exception:
        raise SetupError(f"Grant this notebook access to the {label} Secret.") from None
    if not isinstance(value, str) or len(value) < 24 or any(c.isspace() for c in value):
        raise SetupError(f"The {label} Secret must contain at least 24 non-space characters.")
    return value


def private_url(raw: str, *, loopback=False, allow_query=False, allow_quick_tunnel=False) -> str:
    try:
        u = urllib.parse.urlsplit(raw)
        local = u.hostname in ("127.0.0.1", "localhost", "::1")
        if not u.hostname or u.username or u.password or (u.query and not allow_query) or u.fragment:
            raise ValueError()
        if u.scheme != "https" and not (loopback and local and u.scheme == "http"):
            raise ValueError()
        if (u.hostname == "trycloudflare.com" or u.hostname.endswith(".trycloudflare.com")) and not allow_quick_tunnel:
            raise SetupError("Quick Tunnels do not support relay SSE. Use a stable authenticated HTTPS relay.")
        return raw.rstrip("/")
    except SetupError:
        raise
    except Exception:
        raise SetupError("Use an HTTPS URL without credentials or query parameters.") from None


class NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        # Private release redirects must not leak the Authorization header.
        target = private_url(newurl, allow_query=True)
        original = urllib.parse.urlsplit(req.full_url).hostname
        headers = dict(req.headers)
        if urllib.parse.urlsplit(target).hostname != original:
            headers = {k: v for k, v in headers.items() if k.lower() != "authorization"}
        return urllib.request.Request(target, headers=headers)


def release_artifact(repository="gubnota/auteur", tag="", download_token=None, *, require_relay=True) -> dict:
    """Resolve a published release (including prereleases), without executing downloads."""
    if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repository):
        raise SetupError("Use a GitHub owner/repository name.")
    api = f"https://api.github.com/repos/{repository}"

    def request_json(url, asset=False):
        headers = {"User-Agent": "MyMovie-Notebook", "Accept": "application/octet-stream" if asset else "application/vnd.github+json"}
        if download_token:
            headers["Authorization"] = f"Bearer {download_token}"
        try:
            with urllib.request.build_opener(NoRedirect()).open(urllib.request.Request(url, headers=headers), timeout=30) as response:
                data = response.read(2 * 1024 * 1024 + 1)
            if len(data) > 2 * 1024 * 1024:
                raise ValueError()
            return json.loads(data)
        except Exception:
            raise SetupError("Cannot read the GitHub release metadata. Check the repository/tag and Internet access. Public releases need no token; private releases need download access.") from None

    releases = [request_json(f"{api}/releases/tags/{urllib.parse.quote(tag, safe='')}")] if tag else request_json(f"{api}/releases?per_page=20")
    if not isinstance(releases, list):
        raise SetupError("Unexpected GitHub release list.")
    archive_name = "mymovie-agent-linux-x86_64.tar.gz"
    for release in releases:
        if not isinstance(release, dict) or release.get("draft") or not release.get("published_at"):
            continue
        assets = {a.get("name"): a for a in release.get("assets", []) if isinstance(a, dict)}
        archive = assets.get(archive_name)
        metadata = assets.get("mymovie-agent-linux-x86_64.json")
        if not archive or not metadata:
            continue
        # Construct API URLs ourselves so a release cannot forward credentials to an arbitrary host.
        if any(type(a.get("id")) is not int for a in [archive, metadata]):
            raise SetupError("Invalid release asset IDs.")
        manifest = request_json(f"{api}/releases/assets/{metadata['id']}", asset=True)
        if not isinstance(manifest, dict) or manifest.get("target") != "x86_64-unknown-linux-gnu":
            raise SetupError("Release metadata does not describe the Linux x86_64 agent.")
        digest = manifest.get("sha256", "")
        if not isinstance(digest, str) or not re.fullmatch(r"[a-fA-F0-9]{64}", digest):
            raise SetupError("Release metadata has no valid archive SHA256.")
        github_digest = archive.get("digest")
        if github_digest and (not isinstance(github_digest, str) or github_digest.lower() != f"sha256:{digest.lower()}"):
            raise SetupError("GitHub and release metadata checksums disagree; nothing was installed.")
        relay_digest = manifest.get("relayBinarySha256", "")
        if require_relay and (not isinstance(relay_digest, str) or not re.fullmatch(r"[a-fA-F0-9]{64}", relay_digest)):
            raise SetupError("This published agent archive lacks mymovie-relay for local Cloudflare mode. Publish a new version from the updated CI, or upload a current agent+relay archive with its SHA256. Changing tokens cannot fix an old archive.")
        return {"source": f"{api}/releases/assets/{archive['id']}", "sha256": digest.lower(), "tag": release.get("tag_name", "")}
    raise SetupError("No published release contains the Linux x86_64 agent and metadata. A Git tag alone does not publish artifacts; wait for the release build, or supply an archive and SHA256.")


def install_artifact(source: str, digest: str, destination: Path, download_token=None, *, require_relay=False) -> Path:
    if platform.system() != "Linux" or platform.machine() not in ("x86_64", "AMD64"):
        raise SetupError("This artifact requires Linux x86_64 (Colab/Kaggle GPU runtime).")
    if not re.fullmatch(r"[a-fA-F0-9]{64}", digest):
        raise SetupError("Copy the trusted release SHA256 before installing the artifact.")
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    temporary = tempfile.TemporaryDirectory(dir=destination)
    try:
        archive = Path(temporary.name) / "agent.tar.gz"
        try:
            if source.startswith("https://"):
                url = private_url(source)
                headers = {"User-Agent": "MyMovie-Notebook", "Accept": "application/octet-stream"}
                if download_token:
                    headers["Authorization"] = f"Bearer {download_token}"
                stream = urllib.request.build_opener(NoRedirect()).open(urllib.request.Request(url, headers=headers), timeout=30)
            else:
                stream = Path(source).open("rb")
            total = 0
            h = hashlib.sha256()
            with stream, archive.open("wb") as out:
                while chunk := stream.read(1024 * 1024):
                    total += len(chunk)
                    if total > MAX_ARTIFACT:
                        raise SetupError("Artifact exceeds the download size limit.")
                    h.update(chunk)
                    out.write(chunk)
        except SetupError:
            raise
        except Exception:
            raise SetupError("Artifact unavailable. Check Internet access, release access, or upload the archive locally.") from None
        if h.hexdigest() != digest.lower():
            raise SetupError("Artifact SHA256 mismatch; nothing was executed.")
        try:
            with tarfile.open(archive, "r:gz") as tar:
                members = tar.getmembers()
                if len(members) > 128 or sum(m.size for m in members) > MAX_ARTIFACT:
                    raise SetupError("Unexpected artifact contents.")
                binaries = []; auxiliary = {}; seen = set()
                for member in members:
                    path = Path(member.name)
                    if path.is_absolute() or ".." in path.parts or member.issym() or member.islnk() or not (member.isfile() or member.isdir()):
                        raise SetupError("Unsafe artifact archive; nothing was extracted.")
                    if member.size > MAX_BINARY:
                        raise SetupError("Artifact member exceeds the size limit.")
                    if member.name in seen:
                        raise SetupError("Duplicate artifact member.")
                    seen.add(member.name)
                    if member.name == "mymovie-agent" and member.isfile():
                        binaries.append(member)
                    elif member.isfile() and (member.name in {"mymovie-relay", "mymovie_workflow_compiler.py", "workflow-compiler-requirements.txt", "mymovie_runtime.py", "prepare-workflow-profile.py", "WORKFLOW_EXAMPLES.md", "sd15-model.json", "rembg-plugin.json", "sd15-t4_api.json", "sd15-t4_editor.json", "mask-canary.template.json", "rembg-u2netp.model.json", "mymovie-rembg.plugin.json", "mask-canary_api.json", "mask-canary_editor.json", "rembg-u2netp_api.json", "rembg-u2netp_editor.json"} or re.fullmatch(r"mymovie-rembg-[a-f0-9]{40,64}\.tar\.gz", member.name)):
                        if member.name in auxiliary: raise SetupError("Duplicate runtime helper.")
                        auxiliary[member.name] = tar.extractfile(member).read(MAX_BINARY+1)
                if len(binaries) != 1:
                    raise SetupError("Artifact must contain exactly one mymovie-agent binary.")
                data = tar.extractfile(binaries[0]).read(MAX_BINARY + 1)
        except SetupError:
            raise
        except Exception:
            raise SetupError("Artifact is not a valid agent archive.") from None
        # ELF64 little-endian x86_64: reject a GUI/Mach-O/wrong-target artifact early.
        if data[:6] != b"\x7fELF\x02\x01" or data[18:20] != b"\x3e\x00":
            raise SetupError("The artifact binary is not Linux x86_64 ELF.")
        relay = auxiliary.get("mymovie-relay")
        if require_relay and relay is None:
            raise SetupError("This archive lacks mymovie-relay. Use a current agent+relay archive for local Cloudflare mode.")
        if relay is not None and (relay[:6] != b"\x7fELF\x02\x01" or relay[18:20] != b"\x3e\x00"):
            raise SetupError("The relay binary is not Linux x86_64 ELF.")
        final_dir = destination / digest.lower()
        final_dir.mkdir(exist_ok=True)
        final = final_dir / "mymovie-agent"
        staged = Path(temporary.name) / "mymovie-agent"
        staged.write_bytes(data)
        staged.chmod(0o700)
        os.replace(staged, final)
        for name, contents in auxiliary.items():
            path = final_dir / name
            if path.is_symlink(): raise SetupError("Symlink runtime helper destination.")
            staged = Path(temporary.name) / name
            staged.write_bytes(contents); staged.chmod(0o700 if name == "mymovie-relay" else 0o600); os.replace(staged,path)
        return final
    finally:
        temporary.cleanup()


def gpu_indices() -> list[int]:
    try:
        result = subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10, check=True)
        indices = [int(line.strip()) for line in result.stdout.splitlines() if line.strip()]
        if len(set(indices)) != len(indices):
            raise ValueError()
        return indices
    except Exception:
        raise SetupError("No accessible NVIDIA GPU. Enable a GPU runtime and retry.") from None


def execution_device(mode="auto", count=1):
    """CPU uses one worker; auto uses up to the requested number of CUDA GPUs."""
    if mode not in ("auto", "cpu", "cuda") or count not in (1, 2):
        raise SetupError("Choose DEVICE auto, cpu or cuda and GPU_COUNT 1 or 2.")
    if mode == "cpu":
        return "cpu", 1
    try:
        devices = gpu_indices()
    except SetupError:
        devices = []
    if mode == "cuda" and len(devices) < count:
        raise SetupError("Enable the requested GPU runtime or select DEVICE = 'auto' for CPU fallback.")
    return ("cuda", min(count, len(devices))) if devices else ("cpu", 1)


def comfy_info(url: str) -> dict:
    try:
        with urllib.request.urlopen(f"{url}/system_stats", timeout=5) as response:
            stats = json.load(response)
        with urllib.request.urlopen(f"{url}/object_info", timeout=10) as response:
            registry = json.load(response)
        if not isinstance(registry, dict) or not registry:
            raise ValueError()
        return {"stats": stats, "nodeCount": len(registry)}
    except Exception:
        raise SetupError("ComfyUI is not ready on its loopback port.") from None


def plans(count: int, prefix: str, root: Path, first_port=8188) -> list[dict]:
    if count not in (1, 2) or not re.fullmatch(r"[a-zA-Z0-9_-]{1,80}", prefix):
        raise SetupError("Choose 1 or 2 GPUs and a simple node prefix (letters/digits/hyphens/underscores).")
    if not 1024 <= first_port <= 65534:
        raise SetupError("Choose two available non-privileged ports.")
    return [{"gpu": i, "port": first_port + i, "nodeId": f"{prefix}-gpu{i}", "root": Path(root) / f"gpu{i}"} for i in range(count)]


class NotebookRunner:
    def __init__(self, binary: Path, relay: str, root: Path, count=1, prefix=None, first_port=8188, allow_quick_tunnel=False, cpu=False):
        self.binary = Path(binary).resolve()
        self.relay = private_url(relay, loopback=True, allow_quick_tunnel=allow_quick_tunnel)
        self.root = Path(root).resolve()
        self.root.mkdir(parents=True, exist_ok=True)
        # A fresh session gets a fresh node ID. Keep this object for stop/start with its receipts.
        self.prefix = prefix or f"notebook-{uuid.uuid4().hex[:10]}"
        self.cpu = cpu
        self.workers = plans(1 if cpu else count, self.prefix, self.root, first_port)
        if cpu:
            self.workers[0].update(gpu=None, nodeId=f"{self.prefix}-cpu0", root=self.root / "cpu0")
        self.children = []
        self.comfy_processes = []
        self.supervisor = self.root / "supervisor"
        self.lock = None
        self.started = None
        self.public_relay = None

    def configuration(self) -> dict:
        return {"relay": self.public_relay or self.relay, "nodes": [{"nodeId": w["nodeId"], "comfyUrl": f'http://127.0.0.1:{w["port"]}', "gpu": w["gpu"]} for w in self.workers], "relayAllowlist": ",".join(w["nodeId"] for w in self.workers)}

    def _spawn(self, args, env, cwd=None, log=None):
        # Raw subprocess logs are never copied into shared notebook outputs.
        owned_log = None
        try:
            if log is None:
                path = self.root / f"process-{len(self.children)}.log"
                path.touch(mode=0o600); path.chmod(0o600)
                owned_log = path.open("a"); log = owned_log
            child = subprocess.Popen(args, cwd=cwd, env=env, stdout=log, stderr=log, start_new_session=True)
        except Exception:
            raise SetupError("Unable to start the client. Check Linux runtime compatibility and executable permissions.") from None
        finally:
            if owned_log: owned_log.close()
        self.children.append(child)
        return child

    def _start_relay(self, token, desktop_token, port):
        if not isinstance(desktop_token, str) or not re.fullmatch(r"[!-~]{24,512}", desktop_token) or token == desktop_token:
            raise SetupError("Use a distinct desktop token of 24–512 printable characters.")
        if not isinstance(port, int) or not 1024 <= port <= 65535 or port in [w["port"] for w in self.workers]:
            raise SetupError("Choose an unused relay port from 1024 to 65535, separate from ComfyUI.")
        if self.relay != f"http://127.0.0.1:{port}":
            raise SetupError("Local relay URL must match http://127.0.0.1:RELAY_PORT.")
        binary = self.binary.parent / "mymovie-relay"
        if not binary.is_file():
            raise SetupError("Install a current client archive containing mymovie-relay for local tunnel mode.")
        with socket.socket() as sock:
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                raise SetupError("Relay port is occupied. Choose another RELAY_PORT; existing processes are left running.")
        env = dict(os.environ)
        env.update(MYMOVIE_AGENT_TOKEN=token, MYMOVIE_DESKTOP_TOKEN=desktop_token,
                   MYMOVIE_NODE_IDS=self.configuration()["relayAllowlist"], MYMOVIE_RELAY_BIND=f"127.0.0.1:{port}",
                   MYMOVIE_RELAY_DATA=str(self.root / "relay-data"))
        child = self._spawn([str(binary)], env)
        deadline = time.monotonic() + 15
        request = urllib.request.Request(f'{self.relay}/v1/nodes/{self.workers[0]["nodeId"]}', headers={"Authorization": f"Bearer {desktop_token}"})
        while time.monotonic() < deadline:
            if child.poll() is not None:
                raise SetupError("Local relay exited. Check the artifact and relay port.")
            try:
                with urllib.request.urlopen(request, timeout=2) as response:
                    if response.status == 200: return
            except Exception:
                time.sleep(0.25)
        raise SetupError("Local authenticated relay did not become ready.")

    def _start_tunnel(self, binary):
        executable = shutil.which(binary)
        if not executable:
            raise SetupError("Install cloudflared in this notebook before Start, or use an external relay.")
        path = self.root / "cloudflared.log"
        if path.is_symlink(): raise SetupError("Symlink tunnel log destination.")
        env = {k: v for k, v in os.environ.items() if k not in ("MYMOVIE_AGENT_TOKEN", "MYMOVIE_DESKTOP_TOKEN", "MYMOVIE_TRANSFER_MANIFEST")}
        with path.open("w") as log:
            path.chmod(0o600)
            child = self._spawn([executable, "--no-autoupdate", "tunnel", "--url", self.relay], env, log=log)
        deadline = time.monotonic() + 45
        while time.monotonic() < deadline:
            if child.poll() is not None: raise SetupError("Cloudflare tunnel exited. Check notebook Internet access and cloudflared installation.")
            match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com\b", path.read_text(errors="replace")[-65536:])
            if match:
                self.public_relay = match.group(0)
                return
            time.sleep(0.5)
        raise SetupError("Cloudflare did not return a public relay address. Retry Start or use an external relay.")

    def start(self, token: str, *, comfy_root=None, expected_revision=None, attach=False, model_paths=None, timeout=120, desktop_token=None, relay_port=8787, cloudflared=None, python=None, allow_console=False):
        if self.lock or self.children:
            raise SetupError("This runner is already started. Stop it before starting again.")
        if not isinstance(token, str) or not re.fullmatch(r"[!-~]{24,512}", token):
            raise SetupError("Set MYMOVIE_AGENT_TOKEN to 24–512 printable characters without spaces.")
        if cloudflared and desktop_token is None:
            raise SetupError("Local tunnel mode requires the desktop token to start its authenticated relay.")
        self.lock = (self.root / "runner.lock").open("a+")
        try:
            fcntl.flock(self.lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError:
            self.lock.close(); self.lock = None
            raise SetupError("Another runner owns this working directory.") from None
        try:
            if not self.binary.is_file():
                raise SetupError("Install the verified Linux client artifact first.")
            helper = self.binary.parent / "mymovie_runtime.py"
            if not helper.is_file(): raise SetupError("This feature requires the new agent archive containing mymovie_runtime.py; rebuild with CI.")
            repo = Path(comfy_root or "").resolve()
            if not (repo / "folder_paths.py").is_file(): raise SetupError("Choose the actual ComfyUI root, including for Attach.")
            if desktop_token is not None:
                self._start_relay(token, desktop_token, relay_port)
            devices = [None] if self.cpu else gpu_indices()
            if len(devices) < len(self.workers):
                raise SetupError("Two GPUs were requested, but fewer than two are available. Select one GPU or change the runtime.")
            revision = None
            if attach and len(self.workers) == 2:
                raise SetupError("For two GPUs, use managed Start to guarantee separate device assignment; Attach supports one existing server.")
            if not attach:
                repo = Path(comfy_root or "").resolve()
                if not (repo / "main.py").is_file() or not re.fullmatch(r"[a-fA-F0-9]{40}", expected_revision or ""):
                    raise SetupError("Select an existing ComfyUI installation and its full pinned Git revision.")
                try:
                    revision = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True, timeout=5).strip()
                    dirty = subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain", "--untracked-files=no"], text=True, timeout=5).strip()
                except Exception:
                    raise SetupError("Unable to verify the ComfyUI revision.") from None
                if revision != expected_revision or dirty:
                    raise SetupError("ComfyUI does not match the clean pinned revision.")
                if model_paths and not Path(model_paths).is_file():
                    raise SetupError("The selected extra-model-paths YAML is missing.")
                for w in self.workers:
                    with socket.socket() as sock:
                        if sock.connect_ex(("127.0.0.1", w["port"])) == 0:
                            raise SetupError("A ComfyUI port is occupied. Choose Attach or another port; existing processes are left running.")
                for w, device in zip(self.workers, devices):
                    w["gpu"] = device
                    w["root"].mkdir(parents=True, exist_ok=True)
                    args = [python or sys.executable, str(repo / "main.py"), "--listen", "127.0.0.1", "--port", str(w["port"]), "--disable-auto-launch"]
                    args.extend(["--cpu"] if self.cpu else ["--cuda-device", str(device)])
                    for directory in ("input", "output", "temp", "user"):
                        path = w["root"] / directory; path.mkdir(exist_ok=True)
                        args.extend([f"--{directory}-directory", str(path)])
                    if model_paths:
                        args.extend(["--extra-model-paths-config", str(Path(model_paths).resolve())])
                    # Only agent processes inherit the relay Secret.
                    env = {k: v for k, v in os.environ.items() if k not in ("MYMOVIE_AGENT_TOKEN","MYMOVIE_DESKTOP_TOKEN","MYMOVIE_TRANSFER_MANIFEST")}
                    child = self._spawn(args, env, cwd=repo)
                    self.comfy_processes.append((child, args, env, repo))
            deadline = time.monotonic() + timeout
            for w in self.workers:
                url = f'http://127.0.0.1:{w["port"]}'
                while True:
                    if any(p.poll() is not None for p in self.children):
                        raise SetupError("A ComfyUI process exited before readiness. Inspect process-*.log in the session folder for the cause.")
                    try:
                        info = comfy_info(url)
                        # Managed workers have an explicit CPU or physical GPU assignment.
                        kind = "cpu" if self.cpu else "cuda"
                        matched = [d for d in info["stats"].get("devices", []) if d.get("type") == kind]
                        if len(matched) != 1:
                            raise SetupError(f"Each notebook ComfyUI must expose exactly one {kind.upper()} device.")
                        w["deviceName"] = matched[0].get("name", "")
                        w["readiness"] = info["nodeCount"]
                        break
                    except SetupError as e:
                        if str(e) != "ComfyUI is not ready on its loopback port." or time.monotonic() >= deadline:
                            raise
                        time.sleep(1)
            for w in self.workers:
                data = w["root"] / "agent"; data.mkdir(parents=True, exist_ok=True)
                env = {k: v for k, v in os.environ.items() if k != "MYMOVIE_DESKTOP_TOKEN"}; env["MYMOVIE_AGENT_TOKEN"] = token
                agent_args = [str(self.binary), "serve", "--comfy-url", f'http://127.0.0.1:{w["port"]}', "--relay", self.relay, "--node-id", w["nodeId"], "--data", str(data), "--comfy-root", str(repo), "--python", python or sys.executable, "--runtime-helper", str(helper), "--cache", str(self.root / "dependency-cache")]
                if allow_console: agent_args.append("--allow-console")
                if not attach:
                    self.supervisor.mkdir(parents=True, exist_ok=True)
                    self.supervisor.chmod(0o700)
                    for stale in self.supervisor.glob("*.request.json"): stale.unlink()
                    agent_args.append("--managed-runtime")
                    capabilities = subprocess.run([str(self.binary), "serve", "--help"], capture_output=True, text=True, timeout=10)
                    if "--supervisor" in capabilities.stdout:
                        agent_args.extend(["--supervisor", str(self.supervisor)])
                    else:
                        print("This agent predates guided setup. Publish/install the matching current release for in-app dependency setup and restarts.", flush=True)
                if model_paths: agent_args.extend(["--model-paths",str(Path(model_paths).resolve())])
                compiler = self.binary.parent / "mymovie_workflow_compiler.py"
                if compiler.is_file(): agent_args.extend(["--workflow-compiler", str(compiler)])
                self._spawn(agent_args, env)
            if cloudflared: self._start_tunnel(cloudflared)
            self.started = time.monotonic()
            (self.root / "session.json").write_text(json.dumps({**self.configuration(), "comfyRevision": revision}, indent=2))
        except Exception as e:
            self.stop()
            if isinstance(e, SetupError):
                raise
            raise SetupError("Runner setup failed; check the installation and working directory.") from None
        return self.status()

    def _service_requests(self):
        if not self.supervisor.is_dir(): return
        for path in sorted(self.supervisor.glob("*.request.json")):
            reply = {"ok": False, "error": "Managed restart failed; inspect process logs."}
            try:
                request = json.loads(path.read_text())
                if request.get("action") != "restart-comfyui" or path.name != f'{request.get("id")}.request.json' or not re.fullmatch(r'[a-f0-9-]{36}', request.get("id", "")):
                    raise SetupError("Invalid supervisor request")
                # Check both workers before stopping either; never interrupt a queued/running job.
                for w in self.workers:
                    with urllib.request.urlopen(f'http://127.0.0.1:{w["port"]}/queue', timeout=10) as response:
                        queue = json.load(response)
                    if queue.get("queue_running") or queue.get("queue_pending"):
                        raise SetupError("ComfyUI has running or queued jobs; finish them before restart.")
                for child, _, _, _ in self.comfy_processes:
                    if child.poll() is None:
                        os.killpg(child.pid, signal.SIGTERM)
                        try: child.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            os.killpg(child.pid, signal.SIGKILL); child.wait(timeout=5)
                    self.children.remove(child)
                records = self.comfy_processes; self.comfy_processes = []
                for _, args, env, repo in records:
                    child = self._spawn(args, env, cwd=repo)
                    self.comfy_processes.append((child, args, env, repo))
                deadline = time.monotonic()+120
                for w in self.workers:
                    while True:
                        try: comfy_info(f'http://127.0.0.1:{w["port"]}'); break
                        except SetupError:
                            if time.monotonic()>deadline: raise
                            time.sleep(1)
                    data=w["root"]/"agent"
                    (data/"attestation.json").unlink(missing_ok=True)
                    (data/"runtime-epoch.json").write_text(json.dumps(time.time_ns()))
                (self.root/"dependency-cache"/"restart-required.json").unlink(missing_ok=True)
                reply={"ok": True}
            except SetupError as error:
                reply["error"]=str(error)
            except Exception:
                pass
            finally:
                path.unlink(missing_ok=True)
                dest=path.with_name(path.name.replace(".request.json", ".response.json"))
                temporary=dest.with_suffix(".writing"); temporary.write_text(json.dumps(reply)); temporary.replace(dest)

    def status(self) -> dict:
        alive = sum(p.poll() is None for p in self.children)
        return {**self.configuration(), "processesAlive": alive, "processesStarted": len(self.children), "state": "running" if alive and alive == len(self.children) else "stopped" if not self.children else "attention"}

    def run_for(self, minutes=30):
        if not isinstance(minutes, (int, float)) or not 0 < minutes <= 120:
            raise SetupError("Choose an interactive run of at most 120 minutes.")
        try:
            end = time.monotonic() + minutes * 60
            while time.monotonic() < end:
                self._service_requests()
                if self.status()["state"] != "running":
                    raise SetupError("A worker exited. Keep receipts and inspect connection status before restarting.")
                time.sleep(2)
        finally:
            self.stop()

    def stop(self):
        for p in reversed(self.children):
            if p.poll() is None:
                try:
                    os.killpg(p.pid, signal.SIGTERM)
                    p.wait(timeout=10)
                except ProcessLookupError:
                    pass
                except subprocess.TimeoutExpired:
                    try: os.killpg(p.pid, signal.SIGKILL)
                    except ProcessLookupError: pass
                    p.wait(timeout=5)
        self.children.clear()
        self.comfy_processes.clear()
        self.public_relay = None
        if self.lock:
            fcntl.flock(self.lock, fcntl.LOCK_UN); self.lock.close(); self.lock = None
        return self.status()


if __name__ == "__main__":
    print("Setup helpers loaded. Run all continues with automatic installation and startup.", flush=True)


## 1. Default configuration
Ready to run unchanged. Advanced overrides are optional.


In [ ]:
from pathlib import Path
NOTEBOOK_VERSION = "0.1.21"
HOST = "colab"
DEVICE = "auto"  # auto: CUDA when available, otherwise CPU; or "cpu" / "cuda"
GPU_COUNT = 1  # maximum GPUs in auto; exact GPU count in cuda; ignored for CPU
WORK_DIR = Path("/content/mymovie-session")
RELEASE_REPOSITORY = "gubnota/auteur"
RELEASE_TAG = ""  # optional: pin a published version; blank resolves automatically
ARTIFACT_SOURCE = ""  # optional uploaded archive / HTTPS asset; blank uses GitHub
ARTIFACT_SHA256 = ""  # automatic; required only for a custom ARTIFACT_SOURCE
START_LOCAL_RELAY = True
RELAY_PORT = 8787
RELAY_URL = f"http://127.0.0.1:{RELAY_PORT}"
# External relay override: START_LOCAL_RELAY = False; RELAY_URL = "https://your-relay.example"
ALLOW_QUICK_TUNNEL_POLLING = True
ALLOW_REMOTE_CONSOLE = False  # custom scripts/terminal: replace BOTH shared tokens, then enable
COMFY_ROOT = WORK_DIR / "ComfyUI"  # separate from an existing /content/ComfyUI
COMFY_REVISION = "3541246a9c997f4b8c32f9f88f3c985d02021c79"  # ComfyUI v0.40.0
CLOUDFLARED_SOURCE = "https://github.com/cloudflare/cloudflared/releases/download/2026.10.0/cloudflared-linux-amd64"
CLOUDFLARED_SHA256 = "d33ff2d14475178d2012c2c56beba87389ac5ded27649519f198a7d3134a99db"
CLOUDFLARED_BINARY = str(WORK_DIR / "cloudflared")  # installed automatically
COMFY_PYTHON = str(WORK_DIR / "env/bin/python")
FIRST_PORT = 8190
ATTACH_EXISTING = False  # advanced: existing ComfyUI, one GPU; skip base installation
MODEL_PATHS_YAML = None  # optional extra-model-paths YAML; datasets remain read-only
USE_PRIVATE_DOWNLOAD_SECRET = False

# Public shared test defaults, also populated in Studio Settings → Node access.
# Replace both here and in Studio for private use. Never print token values.
MYMOVIE_AGENT_TOKEN = "wWESjGk2QwVNfaSG9mROZZ_8HQeaLvEjB3IBc-cU2S8"
MYMOVIE_DESKTOP_TOKEN = "gkBVLSAztmflyUU5JtWMQc_3owNVKFvNa2XztixRqbg"
# Optional overrides, disabled by default:
# MYMOVIE_AGENT_TOKEN = os.environ["MYMOVIE_AGENT_TOKEN"]
# MYMOVIE_DESKTOP_TOKEN = os.environ["MYMOVIE_DESKTOP_TOKEN"]
# MYMOVIE_AGENT_TOKEN = secret(HOST, "MYMOVIE_AGENT_TOKEN")
# MYMOVIE_DESKTOP_TOKEN = secret(HOST, "MYMOVIE_DESKTOP_TOKEN")
print(f"MyMovie notebook {NOTEBOOK_VERSION} · CPU/GPU auto detection. No configuration edits required.")


## 2. Automatic installation
This takes several minutes on a fresh runtime. Stages appear below; detailed output stays in `mymovie-session/installation.log`. No model weights are downloaded.


In [ ]:
import platform

_required_bootstrap = ("SetupError", "gpu_indices", "release_artifact", "install_artifact", "prepare_runtime", "NotebookRunner", "secret", "execution_device")
_missing_bootstrap = [name for name in _required_bootstrap if name not in globals()]
if _missing_bootstrap:
    exec(compile('"""Verified Linux agent deployment with CPU or independent notebook GPU workers.\n\nStdlib bootstrap; installs base runtime/tools, never weights or workflow plugins.\n"""\nfrom __future__ import annotations\n\nimport fcntl\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport signal\nimport socket\nimport subprocess\nimport sys\nimport tarfile\nimport tempfile\nimport time\nimport urllib.parse\nimport urllib.request\nimport uuid\n\nMAX_ARTIFACT = 256 * 1024 * 1024\nMAX_BINARY = 128 * 1024 * 1024\n\n\nclass SetupError(RuntimeError):\n    """Public messages are deliberately independent of network/secret exceptions."""\n\n\ndef prepare_runtime(binary, comfy_root, revision, work_dir, *, tunnel_source=None, tunnel_sha256=None, cpu=False):\n    """Provision a base install; reviewed workflow dependencies belong to Studio."""\n    if platform.system() != "Linux" or platform.machine() != "x86_64":\n        raise SetupError("Automatic notebook setup requires a Linux x86_64 runtime.")\n    if not re.fullmatch(r"[a-f0-9]{40}", revision or ""):\n        raise SetupError("The notebook must contain a pinned ComfyUI revision.")\n    root, work = Path(comfy_root).resolve(), Path(work_dir).resolve()\n    work.mkdir(parents=True, exist_ok=True)\n    log = work / "installation.log"\n    log.touch(mode=0o600); log.chmod(0o600)\n    env = {k: v for k, v in os.environ.items() if not k.startswith("MYMOVIE_")}\n    env.update(GIT_TERMINAL_PROMPT="0", PIP_NO_INPUT="1", DEBIAN_FRONTEND="noninteractive")\n\n    def command(args, stage, timeout=1200):\n        print(stage, flush=True)\n        try:\n            with log.open("a") as output:\n                subprocess.run([str(a) for a in args], check=True, stdout=output,\n                               stderr=subprocess.STDOUT, env=env, timeout=timeout)\n        except Exception:\n            raise SetupError(f"{stage} failed. Inspect {log} in this runtime, then retry Run all.") from None\n\n    if not root.exists():\n        root.parent.mkdir(parents=True, exist_ok=True)\n        # Publish a complete checkout; interrupted clones cannot leave a broken target.\n        with tempfile.TemporaryDirectory(prefix=".comfy-", dir=root.parent) as directory:\n            checkout = Path(directory) / "repo"\n            command(["git", "init", checkout], "Preparing base ComfyUI")\n            command(["git", "-C", checkout, "fetch", "--depth=1",\n                     "https://github.com/Comfy-Org/ComfyUI.git", revision], "Downloading pinned ComfyUI")\n            command(["git", "-C", checkout, "checkout", "--detach", "FETCH_HEAD"], "Activating pinned ComfyUI")\n            checkout.rename(root)\n    try:\n        actual = subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL, timeout=10).strip()\n        dirty = subprocess.check_output(["git", "-C", str(root), "status", "--porcelain", "--untracked-files=no"], text=True, stderr=subprocess.DEVNULL, timeout=10).strip()\n    except Exception:\n        raise SetupError(f"Cannot verify {root}. Existing files were left untouched; choose an empty COMFY_ROOT.") from None\n    if actual != revision or dirty or not (root / "main.py").is_file():\n        raise SetupError(f"Existing ComfyUI at {root} conflicts with the pinned clean revision; it was left untouched.")\n\n    requirements = Path(binary).parent / "workflow-compiler-requirements.txt"\n    if not requirements.is_file():\n        raise SetupError("The agent archive must include workflow-compiler-requirements.txt.")\n    python = work / "env/bin/python"\n    marker = work / "base-runtime.json"\n    expected = {"schema": 1, "device": "cpu" if cpu else "cuda", "revision": revision, "python": sys.executable,\n                "pythonVersion": sys.version, "comfyRoot": str(root),\n                "requirements": hashlib.sha256((root / "requirements.txt").read_bytes() + requirements.read_bytes()).hexdigest()}\n    try:\n        cached = python.is_file() and json.loads(marker.read_text()) == expected\n    except (OSError, ValueError):\n        cached = False\n    if not cached:\n        marker.unlink(missing_ok=True)\n        # Reuse the runtime\'s PyTorch without changing notebook packages.\n        # Colab may omit ensurepip; inherit its existing pip instead of requiring apt.\n        command([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", work / "env"], "Creating isolated Python environment")\n        command([python, "-m", "pip", "install", "-r", root / "requirements.txt", "-r", requirements], "Installing ComfyUI and workflow-import dependencies")\n        command([python, "-m", "playwright", "install", "--with-deps", "chromium"], "Installing workflow-import browser")\n    # A marker alone is never readiness; validate the chosen device and the browser\'s OS libraries.\n    try:\n        command([python, "-c", "import sys; sys.path.insert(0, sys.argv[1]); sys.argv=sys.argv[:1]; import torch, aiohttp, folder_paths; "\n                 "from playwright.sync_api import sync_playwright; " +\n                 ("assert torch.ones(1).device.type == \'cpu\'; " if cpu else "assert torch.cuda.is_available(), \'CUDA unavailable\'; ") +\n                 "p=sync_playwright().start(); browser=p.chromium.launch(headless=True, args=[\'--no-sandbox\']); browser.close(); p.stop()", root],\n                f"Verifying {\'CPU\' if cpu else \'CUDA\'} and workflow-import browser", timeout=90)\n    except SetupError:\n        marker.unlink(missing_ok=True)\n        raise\n    temporary = marker.with_suffix(".tmp")\n    temporary.write_text(json.dumps(expected)); temporary.replace(marker)\n    tunnel = None\n    if tunnel_source:\n        private_url(tunnel_source)\n        if not re.fullmatch(r"[a-f0-9]{64}", tunnel_sha256 or ""):\n            raise SetupError("A pinned cloudflared SHA256 is required.")\n        tunnel = work / "cloudflared"\n        if not tunnel.is_file() or hashlib.sha256(tunnel.read_bytes()).hexdigest() != tunnel_sha256:\n            print("Downloading verified cloudflared", flush=True)\n            temporary = tunnel.with_suffix(".part")\n            try:\n                with urllib.request.urlopen(tunnel_source, timeout=60) as response, temporary.open("wb") as output:\n                    size, digest = 0, hashlib.sha256()\n                    while chunk := response.read(1024 * 1024):\n                        size += len(chunk)\n                        if size > MAX_BINARY:\n                            raise SetupError("cloudflared download exceeds the size limit.")\n                        digest.update(chunk); output.write(chunk)\n                if digest.hexdigest() != tunnel_sha256:\n                    raise SetupError("cloudflared checksum mismatch; executable was not installed.")\n                temporary.chmod(0o700); temporary.replace(tunnel)\n            except SetupError:\n                raise\n            except Exception:\n                raise SetupError("cloudflared download failed. Check Internet access and retry Run all.") from None\n            finally:\n                temporary.unlink(missing_ok=True)\n        tunnel.chmod(0o700)\n    print("Base runtime ready. Models and workflow plugins will be installed from Studio.", flush=True)\n    return {"python": str(python), "cloudflared": str(tunnel) if tunnel else None}\n\n\ndef secret(host: str, label: str) -> str:\n    try:\n        if host == "colab":\n            from google.colab import userdata\n            value = userdata.get(label)\n        elif host == "kaggle":\n            from kaggle_secrets import UserSecretsClient\n            value = UserSecretsClient().get_secret(label)\n        else:\n            raise SetupError("Choose colab or kaggle for notebook Secrets.")\n    except Exception:\n        raise SetupError(f"Grant this notebook access to the {label} Secret.") from None\n    if not isinstance(value, str) or len(value) < 24 or any(c.isspace() for c in value):\n        raise SetupError(f"The {label} Secret must contain at least 24 non-space characters.")\n    return value\n\n\ndef private_url(raw: str, *, loopback=False, allow_query=False, allow_quick_tunnel=False) -> str:\n    try:\n        u = urllib.parse.urlsplit(raw)\n        local = u.hostname in ("127.0.0.1", "localhost", "::1")\n        if not u.hostname or u.username or u.password or (u.query and not allow_query) or u.fragment:\n            raise ValueError()\n        if u.scheme != "https" and not (loopback and local and u.scheme == "http"):\n            raise ValueError()\n        if (u.hostname == "trycloudflare.com" or u.hostname.endswith(".trycloudflare.com")) and not allow_quick_tunnel:\n            raise SetupError("Quick Tunnels do not support relay SSE. Use a stable authenticated HTTPS relay.")\n        return raw.rstrip("/")\n    except SetupError:\n        raise\n    except Exception:\n        raise SetupError("Use an HTTPS URL without credentials or query parameters.") from None\n\n\nclass NoRedirect(urllib.request.HTTPRedirectHandler):\n    def redirect_request(self, req, fp, code, msg, headers, newurl):\n        # Private release redirects must not leak the Authorization header.\n        target = private_url(newurl, allow_query=True)\n        original = urllib.parse.urlsplit(req.full_url).hostname\n        headers = dict(req.headers)\n        if urllib.parse.urlsplit(target).hostname != original:\n            headers = {k: v for k, v in headers.items() if k.lower() != "authorization"}\n        return urllib.request.Request(target, headers=headers)\n\n\ndef release_artifact(repository="gubnota/auteur", tag="", download_token=None, *, require_relay=True) -> dict:\n    """Resolve a published release (including prereleases), without executing downloads."""\n    if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", repository):\n        raise SetupError("Use a GitHub owner/repository name.")\n    api = f"https://api.github.com/repos/{repository}"\n\n    def request_json(url, asset=False):\n        headers = {"User-Agent": "MyMovie-Notebook", "Accept": "application/octet-stream" if asset else "application/vnd.github+json"}\n        if download_token:\n            headers["Authorization"] = f"Bearer {download_token}"\n        try:\n            with urllib.request.build_opener(NoRedirect()).open(urllib.request.Request(url, headers=headers), timeout=30) as response:\n                data = response.read(2 * 1024 * 1024 + 1)\n            if len(data) > 2 * 1024 * 1024:\n                raise ValueError()\n            return json.loads(data)\n        except Exception:\n            raise SetupError("Cannot read the GitHub release metadata. Check the repository/tag and Internet access. Public releases need no token; private releases need download access.") from None\n\n    releases = [request_json(f"{api}/releases/tags/{urllib.parse.quote(tag, safe=\'\')}")] if tag else request_json(f"{api}/releases?per_page=20")\n    if not isinstance(releases, list):\n        raise SetupError("Unexpected GitHub release list.")\n    archive_name = "mymovie-agent-linux-x86_64.tar.gz"\n    for release in releases:\n        if not isinstance(release, dict) or release.get("draft") or not release.get("published_at"):\n            continue\n        assets = {a.get("name"): a for a in release.get("assets", []) if isinstance(a, dict)}\n        archive = assets.get(archive_name)\n        metadata = assets.get("mymovie-agent-linux-x86_64.json")\n        if not archive or not metadata:\n            continue\n        # Construct API URLs ourselves so a release cannot forward credentials to an arbitrary host.\n        if any(type(a.get("id")) is not int for a in [archive, metadata]):\n            raise SetupError("Invalid release asset IDs.")\n        manifest = request_json(f"{api}/releases/assets/{metadata[\'id\']}", asset=True)\n        if not isinstance(manifest, dict) or manifest.get("target") != "x86_64-unknown-linux-gnu":\n            raise SetupError("Release metadata does not describe the Linux x86_64 agent.")\n        digest = manifest.get("sha256", "")\n        if not isinstance(digest, str) or not re.fullmatch(r"[a-fA-F0-9]{64}", digest):\n            raise SetupError("Release metadata has no valid archive SHA256.")\n        github_digest = archive.get("digest")\n        if github_digest and (not isinstance(github_digest, str) or github_digest.lower() != f"sha256:{digest.lower()}"):\n            raise SetupError("GitHub and release metadata checksums disagree; nothing was installed.")\n        relay_digest = manifest.get("relayBinarySha256", "")\n        if require_relay and (not isinstance(relay_digest, str) or not re.fullmatch(r"[a-fA-F0-9]{64}", relay_digest)):\n            raise SetupError("This published agent archive lacks mymovie-relay for local Cloudflare mode. Publish a new version from the updated CI, or upload a current agent+relay archive with its SHA256. Changing tokens cannot fix an old archive.")\n        return {"source": f"{api}/releases/assets/{archive[\'id\']}", "sha256": digest.lower(), "tag": release.get("tag_name", "")}\n    raise SetupError("No published release contains the Linux x86_64 agent and metadata. A Git tag alone does not publish artifacts; wait for the release build, or supply an archive and SHA256.")\n\n\ndef install_artifact(source: str, digest: str, destination: Path, download_token=None, *, require_relay=False) -> Path:\n    if platform.system() != "Linux" or platform.machine() not in ("x86_64", "AMD64"):\n        raise SetupError("This artifact requires Linux x86_64 (Colab/Kaggle GPU runtime).")\n    if not re.fullmatch(r"[a-fA-F0-9]{64}", digest):\n        raise SetupError("Copy the trusted release SHA256 before installing the artifact.")\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    temporary = tempfile.TemporaryDirectory(dir=destination)\n    try:\n        archive = Path(temporary.name) / "agent.tar.gz"\n        try:\n            if source.startswith("https://"):\n                url = private_url(source)\n                headers = {"User-Agent": "MyMovie-Notebook", "Accept": "application/octet-stream"}\n                if download_token:\n                    headers["Authorization"] = f"Bearer {download_token}"\n                stream = urllib.request.build_opener(NoRedirect()).open(urllib.request.Request(url, headers=headers), timeout=30)\n            else:\n                stream = Path(source).open("rb")\n            total = 0\n            h = hashlib.sha256()\n            with stream, archive.open("wb") as out:\n                while chunk := stream.read(1024 * 1024):\n                    total += len(chunk)\n                    if total > MAX_ARTIFACT:\n                        raise SetupError("Artifact exceeds the download size limit.")\n                    h.update(chunk)\n                    out.write(chunk)\n        except SetupError:\n            raise\n        except Exception:\n            raise SetupError("Artifact unavailable. Check Internet access, release access, or upload the archive locally.") from None\n        if h.hexdigest() != digest.lower():\n            raise SetupError("Artifact SHA256 mismatch; nothing was executed.")\n        try:\n            with tarfile.open(archive, "r:gz") as tar:\n                members = tar.getmembers()\n                if len(members) > 128 or sum(m.size for m in members) > MAX_ARTIFACT:\n                    raise SetupError("Unexpected artifact contents.")\n                binaries = []; auxiliary = {}; seen = set()\n                for member in members:\n                    path = Path(member.name)\n                    if path.is_absolute() or ".." in path.parts or member.issym() or member.islnk() or not (member.isfile() or member.isdir()):\n                        raise SetupError("Unsafe artifact archive; nothing was extracted.")\n                    if member.size > MAX_BINARY:\n                        raise SetupError("Artifact member exceeds the size limit.")\n                    if member.name in seen:\n                        raise SetupError("Duplicate artifact member.")\n                    seen.add(member.name)\n                    if member.name == "mymovie-agent" and member.isfile():\n                        binaries.append(member)\n                    elif member.isfile() and (member.name in {"mymovie-relay", "mymovie_workflow_compiler.py", "workflow-compiler-requirements.txt", "mymovie_runtime.py", "prepare-workflow-profile.py", "WORKFLOW_EXAMPLES.md", "sd15-model.json", "rembg-plugin.json", "sd15-t4_api.json", "sd15-t4_editor.json", "mask-canary.template.json", "rembg-u2netp.model.json", "mymovie-rembg.plugin.json", "mask-canary_api.json", "mask-canary_editor.json", "rembg-u2netp_api.json", "rembg-u2netp_editor.json"} or re.fullmatch(r"mymovie-rembg-[a-f0-9]{40,64}\\.tar\\.gz", member.name)):\n                        if member.name in auxiliary: raise SetupError("Duplicate runtime helper.")\n                        auxiliary[member.name] = tar.extractfile(member).read(MAX_BINARY+1)\n                if len(binaries) != 1:\n                    raise SetupError("Artifact must contain exactly one mymovie-agent binary.")\n                data = tar.extractfile(binaries[0]).read(MAX_BINARY + 1)\n        except SetupError:\n            raise\n        except Exception:\n            raise SetupError("Artifact is not a valid agent archive.") from None\n        # ELF64 little-endian x86_64: reject a GUI/Mach-O/wrong-target artifact early.\n        if data[:6] != b"\\x7fELF\\x02\\x01" or data[18:20] != b"\\x3e\\x00":\n            raise SetupError("The artifact binary is not Linux x86_64 ELF.")\n        relay = auxiliary.get("mymovie-relay")\n        if require_relay and relay is None:\n            raise SetupError("This archive lacks mymovie-relay. Use a current agent+relay archive for local Cloudflare mode.")\n        if relay is not None and (relay[:6] != b"\\x7fELF\\x02\\x01" or relay[18:20] != b"\\x3e\\x00"):\n            raise SetupError("The relay binary is not Linux x86_64 ELF.")\n        final_dir = destination / digest.lower()\n        final_dir.mkdir(exist_ok=True)\n        final = final_dir / "mymovie-agent"\n        staged = Path(temporary.name) / "mymovie-agent"\n        staged.write_bytes(data)\n        staged.chmod(0o700)\n        os.replace(staged, final)\n        for name, contents in auxiliary.items():\n            path = final_dir / name\n            if path.is_symlink(): raise SetupError("Symlink runtime helper destination.")\n            staged = Path(temporary.name) / name\n            staged.write_bytes(contents); staged.chmod(0o700 if name == "mymovie-relay" else 0o600); os.replace(staged,path)\n        return final\n    finally:\n        temporary.cleanup()\n\n\ndef gpu_indices() -> list[int]:\n    try:\n        result = subprocess.run(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10, check=True)\n        indices = [int(line.strip()) for line in result.stdout.splitlines() if line.strip()]\n        if len(set(indices)) != len(indices):\n            raise ValueError()\n        return indices\n    except Exception:\n        raise SetupError("No accessible NVIDIA GPU. Enable a GPU runtime and retry.") from None\n\n\ndef execution_device(mode="auto", count=1):\n    """CPU uses one worker; auto uses up to the requested number of CUDA GPUs."""\n    if mode not in ("auto", "cpu", "cuda") or count not in (1, 2):\n        raise SetupError("Choose DEVICE auto, cpu or cuda and GPU_COUNT 1 or 2.")\n    if mode == "cpu":\n        return "cpu", 1\n    try:\n        devices = gpu_indices()\n    except SetupError:\n        devices = []\n    if mode == "cuda" and len(devices) < count:\n        raise SetupError("Enable the requested GPU runtime or select DEVICE = \'auto\' for CPU fallback.")\n    return ("cuda", min(count, len(devices))) if devices else ("cpu", 1)\n\n\ndef comfy_info(url: str) -> dict:\n    try:\n        with urllib.request.urlopen(f"{url}/system_stats", timeout=5) as response:\n            stats = json.load(response)\n        with urllib.request.urlopen(f"{url}/object_info", timeout=10) as response:\n            registry = json.load(response)\n        if not isinstance(registry, dict) or not registry:\n            raise ValueError()\n        return {"stats": stats, "nodeCount": len(registry)}\n    except Exception:\n        raise SetupError("ComfyUI is not ready on its loopback port.") from None\n\n\ndef plans(count: int, prefix: str, root: Path, first_port=8188) -> list[dict]:\n    if count not in (1, 2) or not re.fullmatch(r"[a-zA-Z0-9_-]{1,80}", prefix):\n        raise SetupError("Choose 1 or 2 GPUs and a simple node prefix (letters/digits/hyphens/underscores).")\n    if not 1024 <= first_port <= 65534:\n        raise SetupError("Choose two available non-privileged ports.")\n    return [{"gpu": i, "port": first_port + i, "nodeId": f"{prefix}-gpu{i}", "root": Path(root) / f"gpu{i}"} for i in range(count)]\n\n\nclass NotebookRunner:\n    def __init__(self, binary: Path, relay: str, root: Path, count=1, prefix=None, first_port=8188, allow_quick_tunnel=False, cpu=False):\n        self.binary = Path(binary).resolve()\n        self.relay = private_url(relay, loopback=True, allow_quick_tunnel=allow_quick_tunnel)\n        self.root = Path(root).resolve()\n        self.root.mkdir(parents=True, exist_ok=True)\n        # A fresh session gets a fresh node ID. Keep this object for stop/start with its receipts.\n        self.prefix = prefix or f"notebook-{uuid.uuid4().hex[:10]}"\n        self.cpu = cpu\n        self.workers = plans(1 if cpu else count, self.prefix, self.root, first_port)\n        if cpu:\n            self.workers[0].update(gpu=None, nodeId=f"{self.prefix}-cpu0", root=self.root / "cpu0")\n        self.children = []\n        self.comfy_processes = []\n        self.supervisor = self.root / "supervisor"\n        self.lock = None\n        self.started = None\n        self.public_relay = None\n\n    def configuration(self) -> dict:\n        return {"relay": self.public_relay or self.relay, "nodes": [{"nodeId": w["nodeId"], "comfyUrl": f\'http://127.0.0.1:{w["port"]}\', "gpu": w["gpu"]} for w in self.workers], "relayAllowlist": ",".join(w["nodeId"] for w in self.workers)}\n\n    def _spawn(self, args, env, cwd=None, log=None):\n        # Raw subprocess logs are never copied into shared notebook outputs.\n        owned_log = None\n        try:\n            if log is None:\n                path = self.root / f"process-{len(self.children)}.log"\n                path.touch(mode=0o600); path.chmod(0o600)\n                owned_log = path.open("a"); log = owned_log\n            child = subprocess.Popen(args, cwd=cwd, env=env, stdout=log, stderr=log, start_new_session=True)\n        except Exception:\n            raise SetupError("Unable to start the client. Check Linux runtime compatibility and executable permissions.") from None\n        finally:\n            if owned_log: owned_log.close()\n        self.children.append(child)\n        return child\n\n    def _start_relay(self, token, desktop_token, port):\n        if not isinstance(desktop_token, str) or not re.fullmatch(r"[!-~]{24,512}", desktop_token) or token == desktop_token:\n            raise SetupError("Use a distinct desktop token of 24–512 printable characters.")\n        if not isinstance(port, int) or not 1024 <= port <= 65535 or port in [w["port"] for w in self.workers]:\n            raise SetupError("Choose an unused relay port from 1024 to 65535, separate from ComfyUI.")\n        if self.relay != f"http://127.0.0.1:{port}":\n            raise SetupError("Local relay URL must match http://127.0.0.1:RELAY_PORT.")\n        binary = self.binary.parent / "mymovie-relay"\n        if not binary.is_file():\n            raise SetupError("Install a current client archive containing mymovie-relay for local tunnel mode.")\n        with socket.socket() as sock:\n            if sock.connect_ex(("127.0.0.1", port)) == 0:\n                raise SetupError("Relay port is occupied. Choose another RELAY_PORT; existing processes are left running.")\n        env = dict(os.environ)\n        env.update(MYMOVIE_AGENT_TOKEN=token, MYMOVIE_DESKTOP_TOKEN=desktop_token,\n                   MYMOVIE_NODE_IDS=self.configuration()["relayAllowlist"], MYMOVIE_RELAY_BIND=f"127.0.0.1:{port}",\n                   MYMOVIE_RELAY_DATA=str(self.root / "relay-data"))\n        child = self._spawn([str(binary)], env)\n        deadline = time.monotonic() + 15\n        request = urllib.request.Request(f\'{self.relay}/v1/nodes/{self.workers[0]["nodeId"]}\', headers={"Authorization": f"Bearer {desktop_token}"})\n        while time.monotonic() < deadline:\n            if child.poll() is not None:\n                raise SetupError("Local relay exited. Check the artifact and relay port.")\n            try:\n                with urllib.request.urlopen(request, timeout=2) as response:\n                    if response.status == 200: return\n            except Exception:\n                time.sleep(0.25)\n        raise SetupError("Local authenticated relay did not become ready.")\n\n    def _start_tunnel(self, binary):\n        executable = shutil.which(binary)\n        if not executable:\n            raise SetupError("Install cloudflared in this notebook before Start, or use an external relay.")\n        path = self.root / "cloudflared.log"\n        if path.is_symlink(): raise SetupError("Symlink tunnel log destination.")\n        env = {k: v for k, v in os.environ.items() if k not in ("MYMOVIE_AGENT_TOKEN", "MYMOVIE_DESKTOP_TOKEN", "MYMOVIE_TRANSFER_MANIFEST")}\n        with path.open("w") as log:\n            path.chmod(0o600)\n            child = self._spawn([executable, "--no-autoupdate", "tunnel", "--url", self.relay], env, log=log)\n        deadline = time.monotonic() + 45\n        while time.monotonic() < deadline:\n            if child.poll() is not None: raise SetupError("Cloudflare tunnel exited. Check notebook Internet access and cloudflared installation.")\n            match = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com\\b", path.read_text(errors="replace")[-65536:])\n            if match:\n                self.public_relay = match.group(0)\n                return\n            time.sleep(0.5)\n        raise SetupError("Cloudflare did not return a public relay address. Retry Start or use an external relay.")\n\n    def start(self, token: str, *, comfy_root=None, expected_revision=None, attach=False, model_paths=None, timeout=120, desktop_token=None, relay_port=8787, cloudflared=None, python=None, allow_console=False):\n        if self.lock or self.children:\n            raise SetupError("This runner is already started. Stop it before starting again.")\n        if not isinstance(token, str) or not re.fullmatch(r"[!-~]{24,512}", token):\n            raise SetupError("Set MYMOVIE_AGENT_TOKEN to 24–512 printable characters without spaces.")\n        if cloudflared and desktop_token is None:\n            raise SetupError("Local tunnel mode requires the desktop token to start its authenticated relay.")\n        self.lock = (self.root / "runner.lock").open("a+")\n        try:\n            fcntl.flock(self.lock, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except OSError:\n            self.lock.close(); self.lock = None\n            raise SetupError("Another runner owns this working directory.") from None\n        try:\n            if not self.binary.is_file():\n                raise SetupError("Install the verified Linux client artifact first.")\n            helper = self.binary.parent / "mymovie_runtime.py"\n            if not helper.is_file(): raise SetupError("This feature requires the new agent archive containing mymovie_runtime.py; rebuild with CI.")\n            repo = Path(comfy_root or "").resolve()\n            if not (repo / "folder_paths.py").is_file(): raise SetupError("Choose the actual ComfyUI root, including for Attach.")\n            if desktop_token is not None:\n                self._start_relay(token, desktop_token, relay_port)\n            devices = [None] if self.cpu else gpu_indices()\n            if len(devices) < len(self.workers):\n                raise SetupError("Two GPUs were requested, but fewer than two are available. Select one GPU or change the runtime.")\n            revision = None\n            if attach and len(self.workers) == 2:\n                raise SetupError("For two GPUs, use managed Start to guarantee separate device assignment; Attach supports one existing server.")\n            if not attach:\n                repo = Path(comfy_root or "").resolve()\n                if not (repo / "main.py").is_file() or not re.fullmatch(r"[a-fA-F0-9]{40}", expected_revision or ""):\n                    raise SetupError("Select an existing ComfyUI installation and its full pinned Git revision.")\n                try:\n                    revision = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True, timeout=5).strip()\n                    dirty = subprocess.check_output(["git", "-C", str(repo), "status", "--porcelain", "--untracked-files=no"], text=True, timeout=5).strip()\n                except Exception:\n                    raise SetupError("Unable to verify the ComfyUI revision.") from None\n                if revision != expected_revision or dirty:\n                    raise SetupError("ComfyUI does not match the clean pinned revision.")\n                if model_paths and not Path(model_paths).is_file():\n                    raise SetupError("The selected extra-model-paths YAML is missing.")\n                for w in self.workers:\n                    with socket.socket() as sock:\n                        if sock.connect_ex(("127.0.0.1", w["port"])) == 0:\n                            raise SetupError("A ComfyUI port is occupied. Choose Attach or another port; existing processes are left running.")\n                for w, device in zip(self.workers, devices):\n                    w["gpu"] = device\n                    w["root"].mkdir(parents=True, exist_ok=True)\n                    args = [python or sys.executable, str(repo / "main.py"), "--listen", "127.0.0.1", "--port", str(w["port"]), "--disable-auto-launch"]\n                    args.extend(["--cpu"] if self.cpu else ["--cuda-device", str(device)])\n                    for directory in ("input", "output", "temp", "user"):\n                        path = w["root"] / directory; path.mkdir(exist_ok=True)\n                        args.extend([f"--{directory}-directory", str(path)])\n                    if model_paths:\n                        args.extend(["--extra-model-paths-config", str(Path(model_paths).resolve())])\n                    # Only agent processes inherit the relay Secret.\n                    env = {k: v for k, v in os.environ.items() if k not in ("MYMOVIE_AGENT_TOKEN","MYMOVIE_DESKTOP_TOKEN","MYMOVIE_TRANSFER_MANIFEST")}\n                    child = self._spawn(args, env, cwd=repo)\n                    self.comfy_processes.append((child, args, env, repo))\n            deadline = time.monotonic() + timeout\n            for w in self.workers:\n                url = f\'http://127.0.0.1:{w["port"]}\'\n                while True:\n                    if any(p.poll() is not None for p in self.children):\n                        raise SetupError("A ComfyUI process exited before readiness. Inspect process-*.log in the session folder for the cause.")\n                    try:\n                        info = comfy_info(url)\n                        # Managed workers have an explicit CPU or physical GPU assignment.\n                        kind = "cpu" if self.cpu else "cuda"\n                        matched = [d for d in info["stats"].get("devices", []) if d.get("type") == kind]\n                        if len(matched) != 1:\n                            raise SetupError(f"Each notebook ComfyUI must expose exactly one {kind.upper()} device.")\n                        w["deviceName"] = matched[0].get("name", "")\n                        w["readiness"] = info["nodeCount"]\n                        break\n                    except SetupError as e:\n                        if str(e) != "ComfyUI is not ready on its loopback port." or time.monotonic() >= deadline:\n                            raise\n                        time.sleep(1)\n            for w in self.workers:\n                data = w["root"] / "agent"; data.mkdir(parents=True, exist_ok=True)\n                env = {k: v for k, v in os.environ.items() if k != "MYMOVIE_DESKTOP_TOKEN"}; env["MYMOVIE_AGENT_TOKEN"] = token\n                agent_args = [str(self.binary), "serve", "--comfy-url", f\'http://127.0.0.1:{w["port"]}\', "--relay", self.relay, "--node-id", w["nodeId"], "--data", str(data), "--comfy-root", str(repo), "--python", python or sys.executable, "--runtime-helper", str(helper), "--cache", str(self.root / "dependency-cache")]\n                if allow_console: agent_args.append("--allow-console")\n                if not attach:\n                    self.supervisor.mkdir(parents=True, exist_ok=True)\n                    self.supervisor.chmod(0o700)\n                    for stale in self.supervisor.glob("*.request.json"): stale.unlink()\n                    agent_args.append("--managed-runtime")\n                    capabilities = subprocess.run([str(self.binary), "serve", "--help"], capture_output=True, text=True, timeout=10)\n                    if "--supervisor" in capabilities.stdout:\n                        agent_args.extend(["--supervisor", str(self.supervisor)])\n                    else:\n                        print("This agent predates guided setup. Publish/install the matching current release for in-app dependency setup and restarts.", flush=True)\n                if model_paths: agent_args.extend(["--model-paths",str(Path(model_paths).resolve())])\n                compiler = self.binary.parent / "mymovie_workflow_compiler.py"\n                if compiler.is_file(): agent_args.extend(["--workflow-compiler", str(compiler)])\n                self._spawn(agent_args, env)\n            if cloudflared: self._start_tunnel(cloudflared)\n            self.started = time.monotonic()\n            (self.root / "session.json").write_text(json.dumps({**self.configuration(), "comfyRevision": revision}, indent=2))\n        except Exception as e:\n            self.stop()\n            if isinstance(e, SetupError):\n                raise\n            raise SetupError("Runner setup failed; check the installation and working directory.") from None\n        return self.status()\n\n    def _service_requests(self):\n        if not self.supervisor.is_dir(): return\n        for path in sorted(self.supervisor.glob("*.request.json")):\n            reply = {"ok": False, "error": "Managed restart failed; inspect process logs."}\n            try:\n                request = json.loads(path.read_text())\n                if request.get("action") != "restart-comfyui" or path.name != f\'{request.get("id")}.request.json\' or not re.fullmatch(r\'[a-f0-9-]{36}\', request.get("id", "")):\n                    raise SetupError("Invalid supervisor request")\n                # Check both workers before stopping either; never interrupt a queued/running job.\n                for w in self.workers:\n                    with urllib.request.urlopen(f\'http://127.0.0.1:{w["port"]}/queue\', timeout=10) as response:\n                        queue = json.load(response)\n                    if queue.get("queue_running") or queue.get("queue_pending"):\n                        raise SetupError("ComfyUI has running or queued jobs; finish them before restart.")\n                for child, _, _, _ in self.comfy_processes:\n                    if child.poll() is None:\n                        os.killpg(child.pid, signal.SIGTERM)\n                        try: child.wait(timeout=10)\n                        except subprocess.TimeoutExpired:\n                            os.killpg(child.pid, signal.SIGKILL); child.wait(timeout=5)\n                    self.children.remove(child)\n                records = self.comfy_processes; self.comfy_processes = []\n                for _, args, env, repo in records:\n                    child = self._spawn(args, env, cwd=repo)\n                    self.comfy_processes.append((child, args, env, repo))\n                deadline = time.monotonic()+120\n                for w in self.workers:\n                    while True:\n                        try: comfy_info(f\'http://127.0.0.1:{w["port"]}\'); break\n                        except SetupError:\n                            if time.monotonic()>deadline: raise\n                            time.sleep(1)\n                    data=w["root"]/"agent"\n                    (data/"attestation.json").unlink(missing_ok=True)\n                    (data/"runtime-epoch.json").write_text(json.dumps(time.time_ns()))\n                (self.root/"dependency-cache"/"restart-required.json").unlink(missing_ok=True)\n                reply={"ok": True}\n            except SetupError as error:\n                reply["error"]=str(error)\n            except Exception:\n                pass\n            finally:\n                path.unlink(missing_ok=True)\n                dest=path.with_name(path.name.replace(".request.json", ".response.json"))\n                temporary=dest.with_suffix(".writing"); temporary.write_text(json.dumps(reply)); temporary.replace(dest)\n\n    def status(self) -> dict:\n        alive = sum(p.poll() is None for p in self.children)\n        return {**self.configuration(), "processesAlive": alive, "processesStarted": len(self.children), "state": "running" if alive and alive == len(self.children) else "stopped" if not self.children else "attention"}\n\n    def run_for(self, minutes=30):\n        if not isinstance(minutes, (int, float)) or not 0 < minutes <= 120:\n            raise SetupError("Choose an interactive run of at most 120 minutes.")\n        try:\n            end = time.monotonic() + minutes * 60\n            while time.monotonic() < end:\n                self._service_requests()\n                if self.status()["state"] != "running":\n                    raise SetupError("A worker exited. Keep receipts and inspect connection status before restarting.")\n                time.sleep(2)\n        finally:\n            self.stop()\n\n    def stop(self):\n        for p in reversed(self.children):\n            if p.poll() is None:\n                try:\n                    os.killpg(p.pid, signal.SIGTERM)\n                    p.wait(timeout=10)\n                except ProcessLookupError:\n                    pass\n                except subprocess.TimeoutExpired:\n                    try: os.killpg(p.pid, signal.SIGKILL)\n                    except ProcessLookupError: pass\n                    p.wait(timeout=5)\n        self.children.clear()\n        self.comfy_processes.clear()\n        self.public_relay = None\n        if self.lock:\n            fcntl.flock(self.lock, fcntl.LOCK_UN); self.lock.close(); self.lock = None\n        return self.status()\n\n\nif __name__ == "__main__":\n    print("Setup helpers loaded. Run all continues with automatic installation and startup.", flush=True)\n', "<mymovie-bootstrap>", "exec"), globals())

if "runner" in globals() and runner.status()["state"] == "running":
    print("Healthy session already running; reusing its workers and connection address.")
else:
    if "runner" in globals() and runner.status()["state"] == "attention":
        raise SetupError("A session process exited. Inspect process-*.log, then use the Stop cell before retrying.")
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise SetupError("Select a Linux x86_64 notebook runtime.")
    selected_device, worker_count = execution_device(DEVICE, GPU_COUNT)
    print(f"{selected_device.upper()} · {worker_count} worker(s). Resolving the verified agent release…", flush=True)
    download_token = secret(HOST, "MYMOVIE_DOWNLOAD_TOKEN") if USE_PRIVATE_DOWNLOAD_SECRET else None
    try:
        source, digest = ARTIFACT_SOURCE, ARTIFACT_SHA256
        if not source and not digest:
            release = release_artifact(RELEASE_REPOSITORY, RELEASE_TAG, download_token, require_relay=START_LOCAL_RELAY)
            source, digest = release["source"], release["sha256"]
            print(f"Verified release metadata: {release['tag']} · Linux x86_64", flush=True)
        elif not source or not digest:
            raise SetupError("For a custom artifact, set both ARTIFACT_SOURCE and ARTIFACT_SHA256; leave both blank for automatic GitHub setup.")
        binary = install_artifact(source, digest, WORK_DIR / "artifacts", download_token, require_relay=START_LOCAL_RELAY)
    finally:
        download_token = None
    if not ATTACH_EXISTING:
        runtime = prepare_runtime(binary, COMFY_ROOT, COMFY_REVISION, WORK_DIR,
                                  tunnel_source=CLOUDFLARED_SOURCE if START_LOCAL_RELAY else None,
                                  tunnel_sha256=CLOUDFLARED_SHA256, cpu=selected_device == "cpu")
        COMFY_PYTHON = runtime["python"]
        CLOUDFLARED_BINARY = runtime["cloudflared"]
    else:
        COMFY_PYTHON = sys.executable
        CLOUDFLARED_BINARY = "cloudflared"  # advanced Attach requires an installed tunnel
    runner = NotebookRunner(binary, RELAY_URL, WORK_DIR, worker_count, first_port=FIRST_PORT, allow_quick_tunnel=ALLOW_QUICK_TUNNEL_POLLING, cpu=selected_device == "cpu")
    print("Installation complete. Starting the node in the next cell.", flush=True)
    if not START_LOCAL_RELAY:
        print("External relay: add these node IDs to MYMOVIE_NODE_IDS before Start:", runner.configuration()["relayAllowlist"])


## 3. Start ComfyUI and connect the agent
The final output gives the public relay URL and node IDs for Studio.


In [ ]:
if "runner" not in globals():
    raise RuntimeError("Run all notebook cells from the top before starting the node.")
if runner.status()["state"] != "running":
    print("Starting ComfyUI, agent and relay; waiting for readiness and public address…", flush=True)
    runner.start(MYMOVIE_AGENT_TOKEN, comfy_root=COMFY_ROOT, expected_revision=COMFY_REVISION,
                 attach=ATTACH_EXISTING, model_paths=MODEL_PATHS_YAML, python=COMFY_PYTHON, timeout=300,
                 desktop_token=MYMOVIE_DESKTOP_TOKEN if START_LOCAL_RELAY else None,
                 allow_console=ALLOW_REMOTE_CONSOLE, relay_port=RELAY_PORT, cloudflared=CLOUDFLARED_BINARY if START_LOCAL_RELAY else None)
connection = runner.status()
print("\nREADY — Studio → Connections → Colab / Kaggle")
print("Relay URL:", connection["relay"])
for node in connection["nodes"]:
    print("Node ID:", node["nodeId"])
print("Enable Temporary Cloudflare tunnel · polling only. Settings → Node access already has the matching test keys.")
print("Next: Inventory → Review initialization → Initialize workflow. The node stays running until you stop it or the runtime ends.")


## Status
Local process status; Studio reports relay heartbeat and workflow readiness.


In [ ]:
print(json.dumps(runner.status(), indent=2))


## Optional bounded session
Disabled by default. Uncomment to wait up to 30 minutes and then stop owned processes. Interrupting that wait also stops them.


In [ ]:
# runner.run_for(minutes=30)


## Stop — manual action only
Collect results and Drain nodes first. To stop, uncomment the line below and run **only this cell**. Run all leaves it disabled. Attached external ComfyUI is preserved.


In [ ]:
# print(json.dumps(runner.stop(), indent=2))
